# 04 Flat and StructArray Retrieval
Compare flat hybrid passage search with element-level StructArray search. Parent-only EmbeddingList candidates remain routing hints, never citations.

In [ ]:
from agent_workshop_demo.retrieval import InMemoryHybridRetriever
from agent_workshop_demo.sample_data import load_kb_chunks
retriever = InMemoryHybridRetriever(load_kb_chunks())
results = retriever.search('S3 sync Milvus', top_k=3, filters={'department': 'engineering'}, order_by=['updated_at desc', 'priority desc'])
[(item.rank, item.chunk.chunk_id, round(item.hybrid_score, 3)) for item in results]


In [ ]:
from pathlib import Path
from agent_workshop_demo.ingestion import ingest_demo_sources
from agent_workshop_demo.struct_array import InMemoryStructArrayRetriever, StructArrayProfile, build_struct_array_projection, load_projection_manifest
ingestion = ingest_demo_sources(Path('../sample_data/local_docs'), Path('../sample_data/mock_s3'))
projection = build_struct_array_projection(ingestion.kb_chunks, load_projection_manifest(Path('../config/struct_array_projection.json')))
struct = InMemoryStructArrayRetriever(InMemoryHybridRetriever(ingestion.kb_chunks), projection, profile=StructArrayProfile.ELEMENT)
[(item.chunk.chunk_id, item.element_offset) for item in struct.search('S3 sync Milvus', top_k=3, filters={'department': 'engineering', 'is_current': True})]
